<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# Module 2.5: Putting it all Together: An FIR Filter
**Prev: [Sequential Logic](2.4_sequential_logic.ipynb)**<br>
**Next: [Testing with ChiselSim](2.6_chiselsim.ipynb)**

## Motivation
Now that you've learned the basics of Chisel, let's use that knowledge to build a FIR (finite impulse response) filter module! FIR filters are very common in digital signal processing applications. Also, the FIR filter will reappear frequently in module 3, so it's important that you don't filter out this module by skipping ahead! If you are unfamiliar with FIR filters, head over to the article on [trusty Wikipedia](https://en.wikipedia.org/wiki/Finite_impulse_response) to learn more.

## Setup
If a cell fails or hangs, see [Troubleshooting](docs/troubleshooting.md).

In [ ]:
interp.load.module(os.pwd / "source" / "load-ivy.sc")

In [ ]:
import chisel3._
import chisel3.Input // avoid a name clash with almond.input.Input
import chisel3.util._
import chisel3.simulator.PeekPokeAPI._

---
# FIR Filter

The FIR filter you will design performs the following operation.

<img src="images/fir.jpg" alt="FIR filter: a chain of delay registers whose taps are multiplied by the coefficients b0 to b3 and summed into the output" width="720">

Basically, this does a elementwise multiplication of the element of the filter coefficients with the elements of the input signal and outputs the sum (also called a _convolution_).

Or, a signals definition:

$y[n] = b_0 x[n] + b_1 x[n-1] + b_2 x[n-2] + ...$
 - $y[n]$ is the output signal at time $n$
 - $x[n]$ is the input signal
 - $b_i$ are the filter coefficients or impulse response
 - $n-1$, $n-2$, ... are time $n$ delayed by 1, 2, ... cycles
 
## 8-bit Specification

Build a 4-element FIR filter where the four filter coefficients are parameters. A module skeleton and basic tests are provided for you.
Note that both the input and output are 8-bit unsigned integers. You will need to save necessary state (like delayed signal values) using constructs like shift registers. Use the provided testers to check your implementation.
Registers with constant inputs can be created using a `ShiftRegister` of shift value 1, or by using the `RegNext` construct.

Note: for the tests to pass, your registers must be initialized to `0.U` (e.g. with `RegInit` or `RegNext(x, 0.U)`). `simulate` resets the module before the test body runs, which gives these registers their value (see [Module 2.4](2.4_sequential_logic.ipynb)).

In [ ]:
class My4ElementFir(b0: Int, b1: Int, b2: Int, b3: Int) extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(8.W))
    val out = Output(UInt(8.W))
  })

  ???
}

In [ ]:
// Simple sanity check: a element with all zero coefficients should always produce zero
simulate(new My4ElementFir(0, 0, 0, 0)) { c =>
  c.io.in.poke(0.U)
  c.io.out.expect(0.U)
  c.clock.step(1)
  c.io.in.poke(4.U)
  c.io.out.expect(0.U)
  c.clock.step(1)
  c.io.in.poke(5.U)
  c.io.out.expect(0.U)
  c.clock.step(1)
  c.io.in.poke(2.U)
  c.io.out.expect(0.U)
}

In [ ]:
// Simple 4-point moving average
simulate(new My4ElementFir(1, 1, 1, 1)) { c =>
  c.io.in.poke(1.U)
  c.io.out.expect(1.U) // 1, 0, 0, 0
  c.clock.step(1)
  c.io.in.poke(4.U)
  c.io.out.expect(5.U) // 4, 1, 0, 0
  c.clock.step(1)
  c.io.in.poke(3.U)
  c.io.out.expect(8.U) // 3, 4, 1, 0
  c.clock.step(1)
  c.io.in.poke(2.U)
  c.io.out.expect(10.U) // 2, 3, 4, 1
  c.clock.step(1)
  c.io.in.poke(7.U)
  c.io.out.expect(16.U) // 7, 2, 3, 4
  c.clock.step(1)
  c.io.in.poke(0.U)
  c.io.out.expect(12.U) // 0, 7, 2, 3
}

In [ ]:
// Nonsymmetric filter
simulate(new My4ElementFir(1, 2, 3, 4)) { c =>
  c.io.in.poke(1.U)
  c.io.out.expect(1.U) // 1*1, 0*2, 0*3, 0*4
  c.clock.step(1)
  c.io.in.poke(4.U)
  c.io.out.expect(6.U) // 4*1, 1*2, 0*3, 0*4
  c.clock.step(1)
  c.io.in.poke(3.U)
  c.io.out.expect(14.U) // 3*1, 4*2, 1*3, 0*4
  c.clock.step(1)
  c.io.in.poke(2.U)
  c.io.out.expect(24.U) // 2*1, 3*2, 4*3, 1*4
  c.clock.step(1)
  c.io.in.poke(7.U)
  c.io.out.expect(36.U) // 7*1, 2*2, 3*3, 4*4
  c.clock.step(1)
  c.io.in.poke(0.U)
  c.io.out.expect(32.U) // 0*1, 7*2, 2*3, 3*4
}

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-1" />
<label for="check-1"><strong>Solution</strong></label>
<article>
<pre style="background-color:#f7f7f7">
  val x_n1 = RegNext(io.in, 0.U)
  val x_n2 = RegNext(x_n1, 0.U)
  val x_n3 = RegNext(x_n2, 0.U)
  io.out := io.in * b0.U(8.W) + 
    x_n1 * b1.U(8.W) +
    x_n2 * b2.U(8.W) + 
    x_n3 * b3.U(8.W)
</pre></article></div></section></div>

---
# FIR Filter Generator

For this module, we'll be using a slightly modified example from [Module 3.2: Generators: Collection](3.2_collections.ipynb).
If you haven't started Module 3.2, don't worry.
You'll learn about the details of how `MyManyDynamicElementVecFir` works, but the basic idea is that it is a FIR filter generator.

The generator has one parameter: length.
That parameter dictates how many taps the filter has, and the taps are inputs to the hardware `Module`.

The generator has 3 inputs:
* in, the input to the filter
* valid, a boolean that says when the input is valid
* consts, a vector for all the taps

and 1 output:
* out, the filtered input

<img src="images/fir.jpg" alt="FIR filter: a chain of delay registers whose taps are multiplied by the coefficients and summed into the output" style="width:450px;"/>

In [ ]:
class MyManyDynamicElementVecFir(length: Int) extends Module {
  val io = IO(new Bundle {
    val in     = Input(UInt(8.W))
    val valid  = Input(Bool())
    val out    = Output(UInt(8.W))
    val consts = Input(Vec(length, UInt(8.W)))
  })

  // Such concision! You'll learn what all this means later.
  val taps = Seq(io.in) ++ Seq.fill(io.consts.length - 1)(RegInit(0.U(8.W)))
  taps.zip(taps.tail).foreach { case (a, b) => when(io.valid) { b := a } }

  io.out := taps.zip(io.consts).map { case (a, b) => a * b }.reduce(_ + _)
}

visualize(() => new MyManyDynamicElementVecFir(4))

---
# You're done!

[Return to the top.](#top)